Problem Statement & Objective
Context: Diabetes mellitus is a chronic condition affecting millions globally. Early detection allows healthcare providers to intervene with lifestyle modifications and medical management before severe complications occur.

Problem: Clinical diagnostic testing can be resource-intensive. Screening tools that predict risk using non-invasive diagnostic indicators can help prioritize high-risk patients for further evaluation.

Objective: Develop an end-to-end Machine Learning pipeline using diagnostic health measurements to classify patients as high risk or low risk for diabetes.

Primary Business/Clinical Metric: Recall (Sensitivity). In medical screening, minimizing false negatives is vital so that individuals who actually have diabetes are not incorrectly classified as healthy.

In [1]:
# Import core data processing and visualization libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set visual style
sns.set_theme(style="whitegrid")

# Load the Pima Indians Diabetes dataset
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI', 'DiabetesPedigreeFunction', 'Age', 'Outcome']

df = pd.read_csv(url, names=columns)

# Display the first 5 rows
df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 768 entries, 0 to 767
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   Pregnancies               768 non-null    int64  
 1   Glucose                   768 non-null    int64  
 2   BloodPressure             768 non-null    int64  
 3   SkinThickness             768 non-null    int64  
 4   Insulin                   768 non-null    int64  
 5   BMI                       768 non-null    float64
 6   DiabetesPedigreeFunction  768 non-null    float64
 7   Age                       768 non-null    int64  
 8   Outcome                   768 non-null    int64  
dtypes: float64(2), int64(7)
memory usage: 54.1 KB


In [4]:
# Modern pandas syntax for median imputation (avoids FutureWarning)
for col in zero_cols:
    df[col] = df[col].fillna(df[col].median())

# Check target class distribution
print("Target Distribution (Counts):")
print(df['Outcome'].value_counts())
print("\nTarget Distribution (Percentages):")
print(df['Outcome'].value_counts(normalize=True).round(3))

Target Distribution (Counts):
Outcome
0    500
1    268
Name: count, dtype: int64

Target Distribution (Percentages):
Outcome
0    0.651
1    0.349
Name: proportion, dtype: float64


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# Separate features and target
X = df.drop('Outcome', axis=1)
y = df['Outcome']

# Split into 80% training and 20% testing sets with stratification
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Scale features using StandardScaler
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f"Training shape: {X_train_scaled.shape}")
print(f"Testing shape: {X_test_scaled.shape}")

Training shape: (614, 8)
Testing shape: (154, 8)


In [7]:
!pip install xgboost

   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.3/69.5 MB ? eta -:--:--
   ---------------------------------------- 0.5/69.5 MB 1.3 MB/s eta 0:00:56
    --------------------------------------- 1.0/69.5 MB 1.8 MB/s eta 0:00:38
   - -------------------------------------- 2.1/69.5 MB 2.7 MB/s eta 0:00:26
   - -------------------------------------- 3.4/69.5 MB 3.2 MB/s eta 0:00:21
   - -------------------------------------- 3.4/69.5 MB 3.2 MB/s eta 0:00:21
   --- ------------------------------------ 5.5/69.5 MB 3.6 MB/s eta 0:00:18
   --- ------------------------------------ 6.6/69.5 MB 3.7 MB/s eta 0:00:17
   ---- ----------------------------------- 7.9/69.5 MB 4.0 MB/s eta 0:00:16
   ---- ----------------------------------- 8.7/69.5 MB 4.2 MB/s eta 0:00:15
   ------ -----------------

In [8]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import accuracy_score, recall_score, f1_score, roc_auc_score
import pandas as pd

# 1. Initialize models
models = {
    "Logistic Regression": LogisticRegression(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

# 2. Train and evaluate
results = []

for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)
    y_proba = model.predict_proba(X_test_scaled)[:, 1]
    
    acc = accuracy_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc = roc_auc_score(y_test, y_proba)
    
    results.append({
        "Model": name,
        "Accuracy": round(acc, 3),
        "Recall": round(rec, 3),
        "F1-Score": round(f1, 3),
        "ROC-AUC": round(auc, 3)
    })

# 3. Display summary comparison table
results_df = pd.DataFrame(results)
results_df.sort_values(by="Recall", ascending=False)

,Model,Accuracy,Recall,F1-Score,ROC-AUC
1,Random Forest,0.779,0.611,0.660,0.818
2,Gradient Boosting,0.760,0.574,0.626,0.830
0,Logistic Regression,0.708,0.500,0.545,0.813


In [9]:
import joblib
import os

# Create model folder if it doesn't exist
os.makedirs("model", exist_ok=True)

# Save best model (Random Forest) & Scaler
best_model = models["Random Forest"]
joblib.dump(best_model, "model/diabetes_model.pkl")
joblib.dump(scaler, "model/scaler.pkl")

print("Model and scaler successfully saved in 'model/' folder!")

Model and scaler successfully saved in 'model/' folder!
